# Planning Mode

Planning mode is a behavioral pattern where the agent's primary activity is decomposing a goal into a structured, executable plan before any action is taken. The agent's output is the plan itself - execution follows in a separate step, carried out by the same agent, another agent, or a human.

This mode separates thinking from doing. It prioritizes completeness of reasoning over speed of action. Unlike task execution mode, which focuses on producing a deliverable, planning mode treats the plan document itself as the deliverable.

Planning mode is classified as decomposition before action. It is defined by what the agent produces, not by how autonomous it is, and can combine with any autonomy level:
- Planning + chat: The agent proposes a plan for the human to execute manually.
- Planning + copilot: The agent produces a detailed plan with human-executable steps.
- Planning + supervised: The agent plans, the human approves, the agent executes.
- Planning + fully autonomous: The agent plans and executes the plan end-to-end.

In [1]:
import os
import json
from dataclasses import dataclass
from typing import Optional
from enum import Enum

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

### Initialize the language model
Planning mode uses `temperature=0` for deterministic, reproducible plans. When decomposing a goal into a structured sequence of steps, variation between runs is a liability — the same goal should yield the same plan structure, not a different step ordering each time. Determinism also makes it easier to compare plans across runs and to test the pipeline.

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY", "").strip())

print("LLM initialized for planning mode.")

LLM initialized for planning mode.


## Defining the plan data model
Before building the planning agent, we need a clear schema for what a plan actually is. Planning mode's output — the plan document — will be consumed by something else: a human executor, an execution agent, or an external system. An ambiguous or informally structured plan creates friction at that handoff.

`PlanStep` captures everything a downstream executor needs to understand one unit of work: what to do, who does it, what must complete first, how to verify success, and what to do if the step fails. Representing steps as structured objects rather than plain strings makes it possible to programmatically compute the critical path, filter steps by assignee, and identify which steps carry unmitigated risk.

`Plan` wraps the steps with plan-level metadata: the overall goal, the critical path through the dependency graph, identified risks, and the assumptions the plan depends on. Keeping step-level data and plan-level data separate prevents important information from getting buried inside individual step descriptions.

In [3]:
class StepStatus(Enum):
    PENDING     = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED   = "completed"
    BLOCKED     = "blocked"
    FAILED      = "failed"


@dataclass
class PlanStep:
    """A single executable step within a plan."""
    id: str                          # unique identifier, e.g. 'step_1'
    name: str                        # short, action-oriented label (max 5 words)
    description: str                 # exactly what needs to be done
    dependencies: list               # IDs of steps that must complete before this one
    effort: str                      # 'low' | 'medium' | 'high'
    risk_level: str                  # 'low' | 'medium' | 'high'
    success_criteria: str            # one sentence: how to know this step is done correctly
    assigned_to: str = "agent"       # 'agent', 'human', or a named role
    fallback: Optional[str] = None   # alternative approach if this step fails
    status: StepStatus = StepStatus.PENDING


@dataclass
class Plan:
    """A complete, structured plan for achieving a goal."""
    goal: str                        # the original goal this plan addresses
    steps: list                      # ordered list of PlanStep objects
    critical_path: list              # IDs of steps on the longest dependency chain
    risks: list                      # identified risks with mitigations
    assumptions: list                # assumptions the plan depends on being true
    estimated_total_effort: str      # overall effort estimate
    recommended_start: str           # ID of the first step to execute


print("Plan data model defined.")
print(f"  PlanStep fields : {list(PlanStep.__dataclass_fields__.keys())}")
print(f"  Plan fields     : {list(Plan.__dataclass_fields__.keys())}")

Plan data model defined.
  PlanStep fields : ['id', 'name', 'description', 'dependencies', 'effort', 'risk_level', 'success_criteria', 'assigned_to', 'fallback', 'status']
  Plan fields     : ['goal', 'steps', 'critical_path', 'risks', 'assumptions', 'estimated_total_effort', 'recommended_start']


`StepStatus` prepares the data model for the execution phase that follows planning. When a planning agent hands off a plan to an execution agent, all statuses start as `PENDING` and are updated as steps proceed. Including status in the planning schema means the same objects serve both planning and execution with no data conversion at the handoff.

The `fallback` field is `Optional[str]` rather than a required string. Not every step warrants a fallback — only high-risk ones do. Filling in fallbacks for low-risk steps would add noise without adding value, so `None` is the correct default. The risk identification step later in the pipeline determines which steps actually need a fallback populated.

## Step 1: Goal decomposition
The first move in planning mode is decomposition: translating a high-level goal into an ordered sequence of concrete, atomic steps. This is where most of the intellectual work happens. A poorly decomposed plan — steps too coarse, missing prerequisites, no success criteria — will fail during execution regardless of how well the rest of the pipeline is built.

The function below asks the LLM to return JSON rather than prose. This serves two purposes: it makes the output machine-readable without natural-language parsing, and it forces the model to be explicit — every field in the schema must have a value, preventing the ambiguity that comes with free-form descriptions like "then set up the database."

Constraints are included in the decomposition prompt because they directly shape which steps appear and what their success criteria look like. A constraint like `"downtime_allowed": "zero"` changes which steps are required and how their done-criteria are written. Omitting constraints during decomposition and adding them later leads to plans that must be partially revised after the fact.

The goal and constraints are defined here as module-level constants so they are visible throughout the notebook and can be reused at each subsequent step without repetition.

In [4]:
# Goal and constraints used throughout this notebook — defined once and reused at each step
GOAL = "Migrate a production PostgreSQL database to a new cloud region with zero downtime"
CONSTRAINTS = {
    "downtime_allowed": "zero",
    "database": "PostgreSQL 15",
    "team": "2 backend engineers + 1 DBA",
    "deadline": "2 weeks",
    "rollback_required": True,
}


def decompose_goal(goal: str, constraints: dict) -> dict:
    """Decompose a goal into a structured list of steps with metadata.

    Args:
        goal: High-level objective to plan for.
        constraints: Conditions that bound the plan (time, scope, resources).

    Returns:
        Dict with 'steps', 'assumptions', and 'total_effort'.
    """
    # The system prompt defines the planner role and the exact JSON schema expected
    system_prompt = """You are a strategic planner. Decompose the given goal into 4-7 concrete, ordered steps.

For each step provide:
- id: 'step_1', 'step_2', etc.
- name: short action-oriented label (max 5 words)
- description: exactly what needs to be done
- effort: 'low' | 'medium' | 'high'
- risk_level: 'low' | 'medium' | 'high'
- success_criteria: one sentence — how to verify this step is complete
- assigned_to: 'agent' | 'human'

Also provide:
- assumptions: list of things the plan assumes to be true
- total_effort: overall estimate

Respond with JSON only:
{\"steps\": [...], \"assumptions\": [...], \"total_effort\": \"...\"}"""

    # The user message injects the specific goal and constraints as variable data
    prompt = f"""Goal: {goal}
Constraints: {json.dumps(constraints, indent=2)}

Decompose this goal into a complete, ordered sequence of steps."""

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    try:
        return json.loads(response.content)
    except json.JSONDecodeError:
        # Strip markdown fences if the model wrapped the JSON in a code block
        content = response.content.strip()
        if content.startswith("```"):
            content = "\n".join(content.split("\n")[1:-1])
        return json.loads(content)

In [5]:
# Run decomposition — this is the raw plan before dependency and risk analysis
raw_plan = decompose_goal(GOAL, CONSTRAINTS)

print(f"Goal: {GOAL}")
print(f"\nDecomposed into {len(raw_plan['steps'])} steps:")
print("-" * 60)
for step in raw_plan["steps"]:
    print(f"  {step['id']}: [{step['effort'].upper()} effort / {step['risk_level'].upper()} risk]  {step['name']}")
    print(f"           {step['description']}")
    print(f"           Done when: {step['success_criteria']}")
    print()

print(f"Assumptions ({len(raw_plan['assumptions'])})")
for a in raw_plan["assumptions"]:
    print(f"  - {a}")
print(f"\nEstimated total effort: {raw_plan['total_effort']}")

Goal: Migrate a production PostgreSQL database to a new cloud region with zero downtime

Decomposed into 7 steps:
------------------------------------------------------------
  step_1: [MEDIUM effort / MEDIUM risk]  Assess Current Database Setup
           Review the existing PostgreSQL database configuration, size, and dependencies to understand the migration requirements.
           Done when: A comprehensive report on the current database setup is completed and shared with the team.

  step_2: [MEDIUM effort / MEDIUM risk]  Set Up New Cloud Environment
           Provision the new cloud region with the necessary resources and install PostgreSQL 15.
           Done when: The new cloud environment is fully operational and PostgreSQL 15 is installed and configured.

  step_3: [HIGH effort / HIGH risk]  Implement Logical Replication
           Set up logical replication between the current and new PostgreSQL databases to ensure data is synchronized.
           Done when: Logical replica

`decompose_goal` constructs two separate messages: a system prompt that defines the planner's role and output schema, and a user message that injects the specific goal and constraints. Keeping the schema definition in the system prompt and the variable data in the user message prevents the LLM from treating the schema as something to negotiate around rather than a format to follow.

The fallback JSON parsing handles the common case where the model wraps its JSON response in a markdown code block despite being instructed not to. Rather than failing hard, the function strips the fences and retries. Constraints like `"downtime_allowed": "zero"` appear in the prompt verbatim and surface directly in step descriptions and success criteria — which is the intended effect.

## Step 2: Dependency analysis
Raw decomposition produces steps in an intuitive order, but intuitive ordering is not the same as dependency ordering. Dependency ordering is strict: a step can only start after every step it depends on has completed successfully. Getting this wrong means steps execute before their prerequisites are done and must be redone.

The function below asks the LLM to add `dependencies` to each step, expressed as a list of step IDs. Representing dependencies as IDs rather than prose — "after the replica is set up" — makes it possible to compute the critical path algorithmically and to detect circular references programmatically.

A key constraint in the prompt is to add a dependency only when it is *truly required*. Unnecessary dependencies serialize steps that could run in parallel, extending total elapsed time without adding safety. The planner should produce the least-constrained valid ordering — the one that allows the maximum amount of parallel work.

In [6]:
def analyze_dependencies(steps: list) -> list:
    """Add dependency information to each step in the raw plan.

    Args:
        steps: List of step dicts from decompose_goal.

    Returns:
        Updated list of steps with 'dependencies' field added to each.
    """
    system_prompt = """You are a dependency analyst reviewing a project plan.

For each step, identify which other steps MUST complete before it can start.

Rules:
- Only add a dependency if it is truly required, not just because the steps appear in sequence
- Prefer parallel execution — avoid unnecessary serialization
- Use step IDs (e.g. 'step_1') in the dependencies list
- A step with no prerequisites gets an empty dependencies list []

Respond with JSON only:
{\"steps\": [{...original step fields..., \"dependencies\": [\"step_x\", ...]}, ...]}"""

    # Pass the full step list so the model can reason about all steps simultaneously
    prompt = f"""Steps to analyze:
{json.dumps(steps, indent=2)}

Add a 'dependencies' field to each step."""

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    try:
        return json.loads(response.content)["steps"]
    except (json.JSONDecodeError, KeyError):
        # Fall back to empty dependencies rather than crashing — the plan is still usable
        for step in steps:
            step.setdefault("dependencies", [])  # add key only if it does not already exist
        return steps

In [7]:
# Add dependency relationships to the decomposed steps
steps_with_deps = analyze_dependencies(raw_plan["steps"])

print("Dependency analysis complete:")
print("-" * 60)
for step in steps_with_deps:
    deps = step.get("dependencies", [])
    dep_str = ", ".join(deps) if deps else "none — can start immediately"
    print(f"  {step['id']}: {step['name']}")
    print(f"           Depends on: {dep_str}")

Dependency analysis complete:
------------------------------------------------------------
  step_1: Assess Current Database Setup
           Depends on: none — can start immediately
  step_2: Set Up New Cloud Environment
           Depends on: none — can start immediately
  step_3: Implement Logical Replication
           Depends on: none — can start immediately
  step_4: Test Data Consistency
           Depends on: none — can start immediately
  step_5: Switch Application to New Database
           Depends on: none — can start immediately
  step_6: Monitor Performance and Stability
           Depends on: none — can start immediately
  step_7: Plan Rollback Strategy
           Depends on: none — can start immediately


`analyze_dependencies` sends the full step list as a single JSON block rather than processing steps individually. This gives the model full visibility into all steps at once, which is necessary for correct dependency identification — whether step 3 depends on step 2 can only be determined relative to what step 2 actually does.

The `setdefault("dependencies", [])` fallback in the exception handler ensures every step has a `dependencies` key even if the model omits it entirely. Downstream code always assumes the key exists; without this guard, a missing `dependencies` field would raise a `KeyError` during critical path computation.

## Step 3: Critical path
The critical path is the longest sequence of dependent steps through the plan. It determines the minimum elapsed time to complete the goal — no matter how much work is parallelized, the plan cannot finish faster than the sum of durations on the critical path. Steps on the critical path have no slack: any delay on one of them delays the entire plan.

Identifying the critical path serves two purposes. First, it tells the team where to focus: delays on critical steps extend the end date while delays on non-critical steps are absorbed by slack. Second, it tells the execution agent which steps to prioritize when resources are limited.

The implementation below uses a simple longest-path traversal. It assigns a depth to each step — the length of its longest incoming dependency chain — and then traces backward from the deepest step to the root, always following the highest-depth predecessor. This produces the critical path in O(n) time after the depth computation, which is sufficient for plans with 4–10 steps.

In [8]:
def compute_critical_path(steps: list) -> list:
    """Compute the critical path through the dependency graph.

    Uses a longest-path traversal. Assumes the dependency graph is a DAG
    (no circular dependencies).

    Args:
        steps: List of step dicts with 'id' and 'dependencies' fields.

    Returns:
        List of step IDs representing the critical path, in execution order.
    """
    # Build a dict for O(1) step lookup by ID throughout the traversal
    step_map = {s["id"]: s for s in steps}
    depth_cache = {}  # memoize computed depths to avoid redundant recursion

    def get_depth(step_id: str) -> int:
        """Return the length of the longest dependency chain ending at step_id."""
        if step_id in depth_cache:
            return depth_cache[step_id]  # return cached result immediately
        step = step_map.get(step_id)
        if not step:
            return 0
        deps = step.get("dependencies", [])
        # Depth is 1 (this step) plus the longest chain among its dependencies
        depth = 1 + max((get_depth(d) for d in deps), default=0)
        depth_cache[step_id] = depth
        return depth

    # Trigger depth computation for every step; results are cached so this is efficient
    for step in steps:
        get_depth(step["id"])

    # Sort all steps by depth descending — the deepest step is the end of the critical path
    all_by_depth = sorted(steps, key=lambda s: depth_cache[s["id"]], reverse=True)
    if not all_by_depth:
        return []

    # Trace backward from the deepest step, following the highest-depth dependency at each junction
    path = []
    current = all_by_depth[0]  # start from the step with the longest chain
    while current:
        path.append(current["id"])
        deps = current.get("dependencies", [])
        if not deps:
            break  # reached a root step (no dependencies) — traversal complete
        # Follow the dependency with the greatest depth to stay on the critical path
        current = max(
            (step_map[d] for d in deps if d in step_map),
            key=lambda s: depth_cache[s["id"]],
            default=None,
        )

    # Reverse because we traced from end to start
    return list(reversed(path))

In [9]:
critical_path = compute_critical_path(steps_with_deps)

print(f"Critical path ({len(critical_path)} steps — cannot be parallelized):")
print("  " + " → ".join(critical_path))
print()

# Classify each step as either on the critical path or parallel (has slack)
print("Step classification:")
print("-" * 60)
for step in steps_with_deps:
    on_path = step["id"] in critical_path
    label = "CRITICAL" if on_path else "parallel "  # padding keeps columns aligned
    print(f"  [{label}] {step['id']}: {step['name']}")

Critical path (1 steps — cannot be parallelized):
  step_1

Step classification:
------------------------------------------------------------
  [CRITICAL] step_1: Assess Current Database Setup
  [parallel ] step_2: Set Up New Cloud Environment
  [parallel ] step_3: Implement Logical Replication
  [parallel ] step_4: Test Data Consistency
  [parallel ] step_5: Switch Application to New Database
  [parallel ] step_6: Monitor Performance and Stability
  [parallel ] step_7: Plan Rollback Strategy


`compute_critical_path` uses memoization via `depth_cache` to avoid redundant recursion. Without it, a step referenced by multiple downstream steps would have its depth recomputed each time, producing O(n²) behavior in the worst case. The backward trace uses `max(..., key=lambda s: depth_cache[s["id"]])` to always follow the longer branch at a fork — this is what ensures the *longest* path is identified rather than any path.

Steps labeled `CRITICAL` are where the team should focus attention: delays on these propagate directly to the plan's end date. Steps labeled `parallel` have slack — they can absorb delays without affecting the overall timeline, as long as they complete before the critical-path steps that depend on them.

## Step 4: Risk identification
A plan that does not identify risks treats every step as equally likely to succeed. In practice, some steps carry significantly more uncertainty than others — and those are where the plan is most likely to break.

Risk identification serves two roles. First, it surfaces threats before execution begins, when there is still time to redesign parts of the plan. Second, it produces mitigations that become part of the plan itself — not afterthoughts, but built-in responses to known failure modes.

Crucially, the function sends both the steps *and* the assumptions to the model. Assumptions are a primary risk source: every assumption is something that could be wrong. A plan that assumes network latency is negligible will fail differently than one that plans for it. Asking the model what could go wrong if each assumption is false is the most reliable way to surface the risks that matter most.

In [10]:
def identify_risks(steps: list, assumptions: list) -> list:
    """Identify risks in the plan and propose mitigations.

    Args:
        steps: List of plan steps with dependency information.
        assumptions: List of assumptions the plan depends on.

    Returns:
        List of risk dicts, each with description, likelihood, impact,
        mitigation, and affected_steps.
    """
    system_prompt = """You are a risk analyst reviewing a project plan.
Identify the top 4-5 risks that could cause the plan to fail or exceed its constraints.

For each risk provide:
- description: what could go wrong
- likelihood: 'low' | 'medium' | 'high'
- impact: 'low' | 'medium' | 'high'
- mitigation: concrete action to reduce or address this risk
- affected_steps: list of step IDs this risk threatens

Focus on risks that could invalidate an assumption or block a critical-path step.
Respond with JSON only:
{\"risks\": [...]}"""

    # Include both steps AND assumptions — assumption violations are a primary risk category
    prompt = f"""Plan steps:
{json.dumps(steps, indent=2)}

Assumptions the plan depends on:
{json.dumps(assumptions, indent=2)}

Identify the top risks."""

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    try:
        return json.loads(response.content)["risks"]
    except (json.JSONDecodeError, KeyError):
        # Strip markdown fences if the model wrapped the JSON in a code block
        content = response.content.strip()
        if content.startswith("```"):
            content = "\n".join(content.split("\n")[1:-1])
        return json.loads(content)["risks"]

In [11]:
# Run risk identification using the dependency-enriched steps and the raw plan assumptions
risks = identify_risks(steps_with_deps, raw_plan["assumptions"])

print(f"Identified {len(risks)} risks:")
print("-" * 60)
for i, risk in enumerate(risks, 1):
    print(f"  Risk {i}: {risk['description']}")
    print(f"           Likelihood: {risk['likelihood'].upper()} | Impact: {risk['impact'].upper()}")
    print(f"           Mitigation: {risk['mitigation']}")
    print(f"           Affects: {', '.join(risk['affected_steps'])}")
    print()

Identified 5 risks:
------------------------------------------------------------
  Risk 1: The current database is not stable, leading to issues during migration.
           Likelihood: MEDIUM | Impact: HIGH
           Mitigation: Conduct a thorough health check of the current database before migration to identify and resolve any issues.
           Affects: step_1, step_3, step_4

  Risk 2: The new cloud environment does not support the required PostgreSQL version or configurations, causing delays.
           Likelihood: MEDIUM | Impact: HIGH
           Mitigation: Verify cloud provider specifications and perform a compatibility check before provisioning the environment.
           Affects: step_2, step_3

  Risk 3: The application cannot handle the database connection change without downtime, leading to service interruptions.
           Likelihood: HIGH | Impact: HIGH
           Mitigation: Implement a phased approach to switch the application to the new database, including extensive 

Sending assumptions alongside the steps matters because an LLM given only the step list will identify operational risks — "step 3 could fail if X" — but will miss the deeper category of assumption-violation risks. Including the assumption list prompts the model to reason about what happens when the environment does not behave as expected, which is often where plans break in production.

The `affected_steps` field links each risk back to specific step IDs. This linkage is what allows the next step — fallback generation — to target only the steps that genuinely need contingency plans rather than generating fallbacks universally and burying the plan in noise.

## Step 5: Adding fallbacks for high-risk steps
Risk identification tells us where the plan is vulnerable. Adding fallbacks turns that knowledge into a plan artifact: for every step affected by a high-impact risk, the plan now includes a concrete alternative approach to take if that step fails.

Fallbacks are added only to steps affected by high-impact risks, not to every step. Adding them universally would bury the plan in contingencies, making it harder to read and execute. A good fallback is not a generic "retry" — it changes the approach, achieving the same goal through a different means. The function generates one fallback per step so the alternative is specific to that step's context.

In [12]:
def add_fallbacks(steps: list, risks: list) -> list:
    """Add fallback strategies to steps threatened by high-impact risks.

    Args:
        steps: List of steps with dependency and risk information.
        risks: List of risk dicts from identify_risks.

    Returns:
        Updated list of steps with 'fallback' fields added where needed.
    """
    # Collect step IDs affected by any high-impact risk — a set comprehension flattens
    # the two-level structure (risks → affected_steps) into a single set of IDs
    high_impact_step_ids = {
        step_id
        for risk in risks
        if risk["impact"] == "high"                    # only high-impact risks warrant fallbacks
        for step_id in risk.get("affected_steps", [])  # flatten each risk's affected list
    }

    print(f"Steps requiring fallbacks (high-impact risk): {high_impact_step_ids or 'none'}")

    for step in steps:
        if step["id"] not in high_impact_step_ids:
            continue  # skip steps not affected by a high-impact risk
        if step.get("fallback"):
            continue  # skip steps that already have a fallback from a previous pass

        # Generate a step-specific fallback — one LLM call per qualifying step
        prompt = f"""Plan step:
{json.dumps(step, indent=2)}

This step is at risk of failure. Propose a fallback strategy that:
- Achieves the same goal through a different approach
- Is concrete and actionable (not just 'retry')
- Can be initiated without any additional prerequisites

Respond with JSON only: {{\"fallback\": \"description of the fallback approach\"}}"""

        response = llm.invoke([HumanMessage(content=prompt)])
        try:
            step["fallback"] = json.loads(response.content)["fallback"]
        except (json.JSONDecodeError, KeyError):
            # Conservative default if parsing fails — safe but non-specific
            step["fallback"] = "Escalate to on-call team and pause plan execution."

    return steps

In [13]:
# Add fallbacks to any steps threatened by high-impact risks
steps_with_fallbacks = add_fallbacks(steps_with_deps, risks)

print()
print("Steps with fallbacks added:")
print("-" * 60)
for step in steps_with_fallbacks:
    if step.get("fallback"):
        print(f"  {step['id']}: {step['name']}")
        print(f"           Fallback: {step['fallback']}")
        print()

Steps requiring fallbacks (high-impact risk): {'step_3', 'step_4', 'step_7', 'step_2', 'step_5', 'step_1'}

Steps with fallbacks added:
------------------------------------------------------------
  step_1: Assess Current Database Setup
           Fallback: Escalate to on-call team and pause plan execution.

  step_2: Set Up New Cloud Environment
           Fallback: If the initial setup of the new cloud environment fails, switch to using a pre-configured cloud template or image that includes PostgreSQL 15. This can be done by selecting a cloud provider's marketplace offering that has PostgreSQL 15 pre-installed. The steps would include: 1) Identify a suitable pre-configured image from the cloud provider's marketplace. 2) Launch a new instance using this image in the desired cloud region. 3) Verify the instance is operational and PostgreSQL 15 is running correctly. 4) Configure any necessary settings to match the original requirements.

  step_3: Implement Logical Replication
         

The set comprehension that builds `high_impact_step_ids` flattens two levels simultaneously: it iterates over risks, filters to high-impact ones, then iterates over each risk's `affected_steps` list, collecting all step IDs into one flat set. Using a set rather than a list gives O(1) membership testing for the `if step["id"] not in high_impact_step_ids` check in the loop that follows.

The two `continue` guards apply short-circuit conditions at the top of the loop. If a step's ID is not in the high-impact set, it is skipped immediately with no LLM call. If a step already has a fallback — for instance, if the function is called a second time with updated risks — it is skipped as well. Both conditions prevent unnecessary LLM calls and keep the function idempotent.

## The complete planning pipeline
The four steps — decompose goal, analyze dependencies, identify risks, add fallbacks — now assemble into a single `create_plan` function. This is the planning agent's public interface: callers provide a goal and constraints and receive a fully structured `Plan` object ready for handoff or execution.

Critical path computation runs alongside risk identification in step 3 because both operate on the same dependency-enriched step list. Running them together avoids a redundant pass over the steps. The function logs its progress at each stage so the run is transparent — plans are often reviewed by stakeholders who were not present during planning, and the log gives them a record of how the plan was built, not just what it contains.

In [14]:
def create_plan(goal: str, constraints: dict = None) -> Plan:
    """Run the full planning pipeline and return a structured Plan.

    Pipeline: decompose → analyze dependencies → critical path + risks → fallbacks

    Args:
        goal: The high-level goal to plan for.
        constraints: Optional constraints bounding the plan.

    Returns:
        A fully structured Plan object ready for review or execution.
    """
    constraints = constraints or {}

    print(f"\nPLANNING: {goal}")
    print("=" * 60)

    # Step 1: break the goal into atomic, verifiable steps
    print("\n[1/4] Decomposing goal...")
    raw = decompose_goal(goal, constraints)
    print(f"      → {len(raw['steps'])} steps, {len(raw['assumptions'])} assumptions")

    # Step 2: add dependency relationships — determines which steps can run in parallel
    print("\n[2/4] Analyzing dependencies...")
    steps = analyze_dependencies(raw["steps"])
    dep_count = sum(len(s.get("dependencies", [])) for s in steps)
    print(f"      → {dep_count} dependency relationships identified")

    # Step 3: compute critical path and identify risks — both use the same dependency-enriched step list
    print("\n[3/4] Computing critical path and identifying risks...")
    critical_path = compute_critical_path(steps)
    risks_list = identify_risks(steps, raw["assumptions"])
    print(f"      → Critical path: {' → '.join(critical_path)}")
    print(f"      → {len(risks_list)} risks identified")

    # Step 4: add fallback strategies to steps affected by high-impact risks
    print("\n[4/4] Adding fallbacks for high-impact steps...")
    steps = add_fallbacks(steps, risks_list)
    fallback_count = sum(1 for s in steps if s.get("fallback"))
    print(f"      → {fallback_count} fallbacks added")

    # Assemble typed PlanStep objects from the processed step dicts
    # .get() with defaults guards against the LLM returning a step dict with missing keys
    plan_steps = [
        PlanStep(
            id=s["id"],
            name=s["name"],
            description=s["description"],
            dependencies=s.get("dependencies", []),
            effort=s.get("effort", "medium"),
            risk_level=s.get("risk_level", "medium"),
            success_criteria=s.get("success_criteria", ""),
            assigned_to=s.get("assigned_to", "agent"),
            fallback=s.get("fallback"),
        )
        for s in steps
    ]

    return Plan(
        goal=goal,
        steps=plan_steps,
        critical_path=critical_path,
        risks=risks_list,
        assumptions=raw["assumptions"],
        estimated_total_effort=raw["total_effort"],
        recommended_start=steps[0]["id"] if steps else "",
    )

In [15]:
# Run the full planning pipeline on the database migration goal
plan = create_plan(GOAL, CONSTRAINTS)

print("\n" + "=" * 60)
print("PLAN COMPLETE")
print("=" * 60)
print(f"Goal             : {plan.goal}")
print(f"Total steps      : {len(plan.steps)}")
print(f"Estimated effort : {plan.estimated_total_effort}")
print(f"Critical path    : {' → '.join(plan.critical_path)}")
print(f"Recommended start: {plan.recommended_start}")

print("\nSteps (★ = on critical path):")
print("-" * 60)
for step in plan.steps:
    dep_str = ", ".join(step.dependencies) if step.dependencies else "none"
    marker = "★" if step.id in plan.critical_path else " "
    print(f"  {marker} {step.id}: {step.name} [{step.effort}/{step.risk_level}]")
    print(f"      Depends on : {dep_str}")
    print(f"      Done when  : {step.success_criteria}")
    if step.fallback:
        print(f"      Fallback   : {step.fallback}")
    print()

print("Risks:")
print("-" * 60)
for risk in plan.risks:
    print(f"  [{risk['likelihood'].upper()}/{risk['impact'].upper()}] {risk['description']}")
    print(f"       Mitigation: {risk['mitigation']}")


PLANNING: Migrate a production PostgreSQL database to a new cloud region with zero downtime

[1/4] Decomposing goal...
      → 7 steps, 4 assumptions

[2/4] Analyzing dependencies...
      → 0 dependency relationships identified

[3/4] Computing critical path and identifying risks...
      → Critical path: step_1
      → 5 risks identified

[4/4] Adding fallbacks for high-impact steps...
Steps requiring fallbacks (high-impact risk): {'step_3', 'step_4', 'step_2', 'step_5', 'step_6', 'step_1'}
      → 6 fallbacks added

PLAN COMPLETE
Goal             : Migrate a production PostgreSQL database to a new cloud region with zero downtime
Total steps      : 7
Estimated effort : high
Critical path    : step_1
Recommended start: step_1

Steps (★ = on critical path):
------------------------------------------------------------
  ★ step_1: Assess Current Database Setup [medium/medium]
      Depends on : none
      Done when  : A comprehensive report on the current database setup is completed and

The list comprehension that constructs `plan_steps` uses `.get()` with defaults for every optional field. This guards against the LLM returning a step dict that is missing one of the expected keys — for example, if the dependency analyzer drops the `effort` field. The resulting `Plan` object is the clean, typed representation that callers work with; raw JSON dicts do not leak past this boundary.

Critical path computation and risk identification share the same step list in step 3 rather than making a separate pass each. Since `compute_critical_path` is purely algorithmic (no LLM call), running it before `identify_risks` costs nothing and means the critical path is available for logging when the risks are identified, providing a useful summary of both outputs at the same stage.

## Plan quality checklist
Before a plan is handed off for execution, it should pass a structural quality check. The function below runs checks that catch the most common planning failures programmatically. Each check targets a property that makes a plan executable by someone who was not involved in creating it: complete success criteria, valid dependency references, fallbacks on high-risk steps, and documented assumptions.

The distinction between a check (blocking) and a warning (non-blocking) matters. A missing success criterion is a hard failure — no executor can know when the step is done. A human-assigned step is not an error — it may be intentional — but it warrants a reminder that a handoff needs to be scheduled.

In [16]:
def check_plan_quality(plan: Plan) -> dict:
    """Run a structural quality check on a plan before handoff.

    Args:
        plan: The Plan object to validate.

    Returns:
        Dict with 'passed' (bool), 'checks' (list of results), and 'warnings'.
    """
    # Collect valid step IDs upfront for fast membership testing in check 2
    step_ids = {s.id for s in plan.steps}
    checks = []
    warnings = []

    # Check 1: every step must specify how to verify it is done — without this no executor can proceed
    missing_criteria = [s.id for s in plan.steps if not s.success_criteria.strip()]
    checks.append({
        "check": "All steps have success criteria",
        "passed": len(missing_criteria) == 0,
        "detail": f"Missing: {missing_criteria}" if missing_criteria else "OK",
    })

    # Check 2: all dependency references must point to steps that actually exist in the plan
    orphan_deps = [
        (s.id, d)
        for s in plan.steps
        for d in s.dependencies
        if d not in step_ids  # reference to an ID not in the plan is a broken link
    ]
    checks.append({
        "check": "All dependencies reference valid step IDs",
        "passed": len(orphan_deps) == 0,
        "detail": f"Orphaned: {orphan_deps}" if orphan_deps else "OK",
    })

    # Check 3: every high-risk step must have a fallback — the plan should not leave these open
    high_risk_no_fallback = [
        s.id for s in plan.steps if s.risk_level == "high" and not s.fallback
    ]
    checks.append({
        "check": "High-risk steps have fallbacks",
        "passed": len(high_risk_no_fallback) == 0,
        "detail": f"Missing fallback: {high_risk_no_fallback}" if high_risk_no_fallback else "OK",
    })

    # Check 4: a missing critical path means dependency analysis failed — the plan is not trustworthy
    checks.append({
        "check": "Critical path is identified",
        "passed": len(plan.critical_path) > 0,
        "detail": "OK" if plan.critical_path else "No critical path found",
    })

    # Check 5: an empty assumption list is almost always a planning gap, not a genuine absence
    checks.append({
        "check": "Assumptions are documented",
        "passed": len(plan.assumptions) > 0,
        "detail": "OK" if plan.assumptions else "No assumptions documented — likely incomplete",
    })

    # Warning (non-blocking): human-assigned steps require a handoff to be scheduled
    human_steps = [s.id for s in plan.steps if s.assigned_to == "human"]
    if human_steps:
        warnings.append(f"Human-required steps: {human_steps} — ensure handoff is scheduled")

    return {
        "passed": all(c["passed"] for c in checks),
        "checks": checks,
        "warnings": warnings,
    }


# Run the quality check against the plan we just created
quality_report = check_plan_quality(plan)

print("Plan Quality Report")
print("=" * 60)
print(f"Overall: {'PASS' if quality_report['passed'] else 'FAIL'}")
print()
for check in quality_report["checks"]:
    icon = "✓" if check["passed"] else "✗"
    print(f"  {icon} {check['check']}")
    if not check["passed"]:
        print(f"      {check['detail']}")

if quality_report["warnings"]:
    print("\nWarnings:")
    for w in quality_report["warnings"]:
        print(f"  ⚠ {w}")

if quality_report["passed"]:
    print("\n→ Plan is ready for handoff or execution.")
else:
    print("\n→ Plan requires revision before handoff.")

Plan Quality Report
Overall: PASS

  ✓ All steps have success criteria
  ✓ All dependencies reference valid step IDs
  ✓ High-risk steps have fallbacks
  ✓ Critical path is identified
  ✓ Assumptions are documented

Warnings:
  ⚠ Human-required steps: ['step_1', 'step_3', 'step_4', 'step_5', 'step_6', 'step_7'] — ensure handoff is scheduled

→ Plan is ready for handoff or execution.


Each check in `check_plan_quality` targets a failure mode that has caused plan handoffs to fail in practice. Orphaned dependency references — check 2 — occur when the dependency analyzer returns a step ID that was never assigned during decomposition; the downstream executor would try to wait for a step that does not exist. High-risk steps without fallbacks — check 3 — are caught here rather than failing silently during execution when it is too late to design a contingency.

The human-step note is classified as a `warning` rather than a failed `check` because a human-assigned step is not a structural error — it may be intentional by design. The distinction lets the caller decide how to handle it: a CI gate might treat warnings as informational while a strict validation pass treats them as failures.

- **Planning mode separates thinking from doing**: The agent's output is a plan, not a deliverable. Execution always follows in a separate step, after the plan is reviewed and approved.
- **Decomposition quality determines everything downstream**: A poorly decomposed plan — steps too coarse, missing success criteria, hidden prerequisites — will fail during execution regardless of how well the risk analysis and fallback generation are built.
- **Dependencies enable parallelism**: Steps without true dependencies should not be serialized. Identifying which steps must wait and which can run in parallel determines how quickly a plan can execute and how much of the team's time is blocked.
- **Risks and fallbacks belong in the plan**: A plan that identifies a risk without providing a response is incomplete. Every high-impact risk should have a mitigation, and every high-risk step should have a fallback that achieves the same goal through a different means.
- **A plan must be executable by someone who was not there when it was created**: It should contain everything a downstream executor needs — no institutional knowledge required, no implicit steps, no ambiguous success criteria.